# Étape 2 — Outil d'annotation des offres

1. Colle une offre → **Pré-remplir avec le modèle**
2. Corrige les champs en suivant `GUIDE_ANNOTATION.md`
3. **Valider et enregistrer** → l'offre est ajoutée à `offres.jsonl` sur ton Google Drive

Les annotations sont sauvegardées sur Drive : tu peux fermer Colab et reprendre plus tard sans rien perdre.

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4.

In [ ]:
# 1. Installation + Google Drive (pour ne jamais perdre tes annotations)
!pip install -q transformers accelerate pydantic

from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = '/content/drive/MyDrive/job-offer-extractor/data'
os.makedirs(DOSSIER, exist_ok=True)
FICHIER = f'{DOSSIER}/offres.jsonl'
print('Les annotations seront enregistrées dans :', FICHIER)

In [ ]:
# 2. Connexion Hugging Face (même méthode que l'étape 1)
from huggingface_hub import login
login()

In [ ]:
# 3. Charger le modèle (1 à 3 min)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)
print('Modèle chargé sur', model.device)

In [ ]:
# 4. Schéma + règles d'annotation
import json
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

TYPES = ['PFE', 'stage', 'SIVP', 'CDI', 'CDD', 'freelance']
MODES = ['présentiel', 'hybride', 'remote']
LANGUES = ['fr', 'en', 'ar']

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'SIVP', 'CDI', 'CDD', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    competences: list[str]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']

SCHEMA = json.dumps(Offre.model_json_schema(), ensure_ascii=False)

REGLES = '''Règles :
- type_contrat = "PFE" si l'offre parle de PFE ou de fin d'études, sinon "stage" pour un stage. Si plusieurs types sont cités, prends le premier. "Saisonnier" pour un stagiaire = "stage".
- remuneration = le montant s'il est donné ; "rémunéré" si l'offre dit seulement rémunéré ; une prime n'est pas un salaire ; "Confidentiel" = null.
- competences = uniquement des compétences techniques ou métier (outils, langages, frameworks, domaines). Jamais d'années d'expérience, de diplôme, de tâches ni de qualités humaines.
- langue_offre = UNE seule valeur, la langue dominante.
- ville et entreprise sont deux champs séparés.
- Si une information est absente, mets null. Ne devine jamais.'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    'Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n'
    + SCHEMA + '\n\n' + REGLES
)

In [ ]:
# 5. Fonctions d'extraction
def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

def extraire(offre):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': offre},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    n = inputs['input_ids'].shape[1]
    return tokenizer.decode(out[0][n:], skip_special_tokens=True)

def compter_annotations():
    if not os.path.exists(FICHIER):
        return 0
    with open(FICHIER, encoding='utf-8') as f:
        return sum(1 for ligne in f if ligne.strip())

In [ ]:
# 6. L'outil d'annotation
import ipywidgets as w
from IPython.display import display
from datetime import date

VIDE = '(vide)'
large = w.Layout(width='100%')

# --- Colonne gauche : l'offre ---
offre = w.Textarea(placeholder="Colle l'offre complète ici", layout=w.Layout(width='100%', height='420px'))
source = w.Text(description='Source', placeholder='TanitJobs, Keejob, LinkedIn…', layout=large)

# --- Colonne droite : les champs ---
poste = w.Text(description='Poste', layout=large)
entreprise = w.Text(description='Entreprise', layout=large)
ville = w.Text(description='Ville', layout=large)
type_contrat = w.Dropdown(description='Contrat', options=[VIDE] + TYPES)
mode_travail = w.Dropdown(description='Mode', options=[VIDE] + MODES)
duree = w.Text(description='Durée (mois)', placeholder='ex. 6')
competences = w.Textarea(description='Compétences', placeholder='une par ligne',
                         layout=w.Layout(width='100%', height='160px'))
remuneration = w.Text(description='Rémunération', layout=large)
langue = w.Dropdown(description='Langue', options=LANGUES)

for champ in [poste, entreprise, ville, type_contrat, mode_travail, duree,
              competences, remuneration, langue, source]:
    champ.style.description_width = '110px'

btn_prefill = w.Button(description='Pré-remplir avec le modèle', button_style='info', icon='magic')
btn_save = w.Button(description='Valider et enregistrer', button_style='success', icon='check')
btn_clear = w.Button(description='Effacer', icon='eraser')
compteur = w.HTML()
sortie = w.Output()

def maj_compteur():
    n = compter_annotations()
    compteur.value = f'<b>{n}</b> offres annotées (objectif : 50)'

def choix(valeur, options):
    return valeur if valeur in options else VIDE

def vide_si_none(v):
    return '' if v is None else str(v)

def pre_remplir(_):
    sortie.clear_output()
    with sortie:
        if not offre.value.strip():
            print('⚠️ Colle d’abord une offre.')
            return
        print('⏳ Le modèle lit l’offre…')
        brut = extraire(offre.value)
        try:
            d = json.loads(nettoyer(brut))
        except json.JSONDecodeError:
            print('❌ Le modèle n’a pas renvoyé de JSON valide. Remplis les champs à la main.')
            print(brut)
            return
        poste.value = vide_si_none(d.get('poste'))
        entreprise.value = vide_si_none(d.get('entreprise'))
        ville.value = vide_si_none(d.get('ville'))
        type_contrat.value = choix(d.get('type_contrat'), TYPES)
        mode_travail.value = choix(d.get('mode_travail'), MODES)
        duree.value = vide_si_none(d.get('duree_mois'))
        comps = d.get('competences') or []
        competences.value = '\n'.join(str(c) for c in comps) if isinstance(comps, list) else str(comps)
        remuneration.value = vide_si_none(d.get('remuneration'))
        langue.value = d.get('langue_offre') if d.get('langue_offre') in LANGUES else 'fr'
        sortie.clear_output()
        print('✅ Pré-rempli. Vérifie et corrige chaque champ avant d’enregistrer.')

def ou_none(texte):
    texte = texte.strip()
    return texte if texte else None

def enregistrer(_):
    sortie.clear_output()
    with sortie:
        if not offre.value.strip():
            print('⚠️ Aucune offre à enregistrer.')
            return
        try:
            duree_val = float(duree.value.replace(',', '.')) if duree.value.strip() else None
        except ValueError:
            print('❌ La durée doit être un nombre (ex. 6) ou vide.')
            return
        data = {
            'poste': poste.value.strip(),
            'entreprise': ou_none(entreprise.value),
            'ville': ou_none(ville.value),
            'type_contrat': None if type_contrat.value == VIDE else type_contrat.value,
            'mode_travail': None if mode_travail.value == VIDE else mode_travail.value,
            'duree_mois': duree_val,
            'competences': [c.strip() for c in competences.value.split('\n') if c.strip()],
            'remuneration': ou_none(remuneration.value),
            'langue_offre': langue.value,
        }
        if not data['poste']:
            print('❌ Le poste est obligatoire.')
            return
        try:
            Offre.model_validate(data)
        except ValidationError as e:
            print('❌ Ne respecte pas le schéma :', e)
            return
        ligne = {
            'offre': offre.value.strip(),
            'extraction': data,
            'source': ou_none(source.value),
            'date_annotation': date.today().isoformat(),
        }
        with open(FICHIER, 'a', encoding='utf-8') as f:
            f.write(json.dumps(ligne, ensure_ascii=False) + '\n')
        print('💾 Enregistrée !')
        effacer(None, garder_message=True)
        maj_compteur()

def effacer(_, garder_message=False):
    for champ in [offre, poste, entreprise, ville, duree, competences, remuneration]:
        champ.value = ''
    type_contrat.value = VIDE
    mode_travail.value = VIDE
    langue.value = 'fr'
    if not garder_message:
        sortie.clear_output()

btn_prefill.on_click(pre_remplir)
btn_save.on_click(enregistrer)
btn_clear.on_click(effacer)
maj_compteur()

gauche = w.VBox([w.HTML('<h3>Offre</h3>'), offre, source, btn_prefill],
                layout=w.Layout(width='48%'))
droite = w.VBox([w.HTML('<h3>Extraction</h3>'), poste, entreprise, ville, type_contrat,
                 mode_travail, duree, competences, remuneration, langue,
                 w.HBox([btn_save, btn_clear])],
                layout=w.Layout(width='50%'))
display(w.VBox([compteur, w.HBox([gauche, droite]), sortie]))

In [ ]:
# 7. Bilan du dataset (à lancer quand tu veux)
from collections import Counter

with open(FICHIER, encoding='utf-8') as f:
    lignes = [json.loads(l) for l in f if l.strip()]

print('Total :', len(lignes))
print('Langues :', Counter(l['extraction']['langue_offre'] for l in lignes))
print('Contrats :', Counter(l['extraction']['type_contrat'] for l in lignes))
print('Sources :', Counter(l['source'] for l in lignes))

textes = [l['offre'] for l in lignes]
doublons = len(textes) - len(set(textes))
print('Doublons :', doublons)

In [ ]:
# Vérification rapide du texte téléchargé
with open(FICHIER, encoding='utf-8') as f:
    L = [json.loads(l) for l in f if l.strip()]
tailles = sorted(len(l['offre']) for l in L)
print('Longueur des offres : min', tailles[0], '| médiane', tailles[len(tailles)//2], '| max', tailles[-1])
for i in (0, 27):
    print('-' * 60, '\n', L[i]['offre'][:350].replace('\n', ' | '))

In [ ]:
# (Ancienne version du nettoyage — remplacée par les cellules 9 à 12 ci-dessous.)
# Elle supprimait la ligne « 500 » du salaire Keejob (« 500 / - 570 TND / Mois ») et réutilisait les étiquettes non corrigées.


In [ ]:
# 9. Corrections des étiquettes (étapes 1 et 2) — offres.jsonl n'est PAS modifié
import json, re, csv
with open(FICHIER, encoding='utf-8') as f:
    DATA = [json.loads(l) for l in f if l.strip()]
T = 'https://www.tanitjobs.com/job/'
M = 'https://mostaql.com/project/'

# Étape 1 — règle « premier type cité » (Temps plein / Temps partiel ne sont pas des types de contrat)
# Aucune offre ne cite SIVP en premier : SIVP n'est donc la valeur d'aucune offre existante.
TYPE_FIX = {
    T+'2057842/': 'CDI',        # CDI, CDD, Temps plein, ..., SIVP, Stage
    T+'2056989/': 'CDI',        # CDI, Freelance / Indépendant, SIVP
    T+'2054141/': 'freelance',  # Temps plein, Temps plein, Freelance / Indépendant
    T+'2054481/': 'freelance',  # Temps plein, Freelance / Indépendant
    T+'2054477/': 'freelance',
    T+'2054483/': 'freelance',
}
# Étape 2 — tâches / activités / livrables retirés des compétences
RETIRER = {
    T+'2037743/': ['automatisation de processus métiers'],
    T+'2049987/': ['analyse fonctionnelle'],
    T+'2055751/': ['tests fonctionnels'],
    T+'2055598/': ['log file analysis'],
    T+'2059099/': ['manual testing', 'application support', 'incident management'],
    T+'1899133/': ['business analysis'],
    M+'1284365': ['Web App', 'وسائل الدفع الإلكتروني'],
    M+'1284316': ['إدارة المشاريع البرمجية'],
    M+'1284264': ['واتساب'],
    M+'1284126': ['نشر التطبيقات'],
    M+'1284427': ['استكشاف الأخطاء وإصلاحها'],
    M+'1284410': ['إنشاء متجر إلكتروني', 'إدارة المتاجر الإلكترونية'],
    M+'1283746': ['Web Dashboard', 'تطبيقات الجوال'],
    M+'1284340': ['Business Workflows'],
}
RENOMMER = {M+'1284340': {'المحاسبة المالية': 'محاسبة مالية'}}  # forme exacte du texte

urls = {l['url'] for l in DATA}
assert set(TYPE_FIX) | set(RETIRER) | set(RENOMMER) <= urls, 'url inconnue'
journal = []
for l in DATA:
    u, e = l['url'], l['extraction']
    if u in TYPE_FIX:
        journal.append((u, 'type_contrat', e['type_contrat'], TYPE_FIX[u]))
        e['type_contrat'] = TYPE_FIX[u]
    for c in RETIRER.get(u, []):
        assert c in e['competences'], (u, c)
    avant = list(e['competences'])
    apres = [RENOMMER.get(u, {}).get(c, c) for c in avant if c not in RETIRER.get(u, [])]
    if apres != avant:
        journal.append((u, 'competences', avant, apres))
        e['competences'] = apres
    Offre.model_validate(e)

print(len(journal), 'modifications :')
for u, champ, av, ap in journal:
    if champ == 'competences':
        print(f"{u} | competences | retiré {[c for c in av if c not in ap]}" + (f" | ajouté {[c for c in ap if c not in av]}" if any(c not in av for c in ap) else ''))
    else:
        print(f'{u} | {champ} | {av} → {ap}')
with open(f'{DOSSIER}/journal_corrections.csv', 'w', encoding='utf-8', newline='') as f:
    w_ = csv.writer(f); w_.writerow(['url', 'champ', 'avant', 'apres'])
    for u, champ, av, ap in journal:
        w_.writerow([u, champ, json.dumps(av, ensure_ascii=False), json.dumps(ap, ensure_ascii=False)])
print('Journal complet (avant/après) :', f'{DOSSIER}/journal_corrections.csv')

In [ ]:
# 10. Nettoyage du texte (étape 3) → offres_propres.jsonl (sans troncature)
import unicodedata
BRUIT_LIGNE = re.compile(r'^(Sauvegarder|Plus de|\d+|candidats\s+pour|\d*\s*posts?\s+ouverts?|\d*\s*postes?\s+ouverts?)$', re.I)
DEBUT_MONTANT = re.compile(r'^\d[\d\s.,]*(\s*-\s*\d[\d\s.,]*)?$')
SUITE_MONTANT = re.compile(r'^(-\s*\d[\d\s.,]*|(TND|DT|€|\$)\b.*)$', re.I)

def nettoyer_offre(texte):
    lignes = [re.sub(r'\s+', ' ', x).strip() for x in texte.split('\n')]
    fus = []
    for x in lignes:
        if not x:
            continue
        # recoller un montant éclaté (Keejob : "500" / "- 570" / "TND / Mois") avant de retirer les nombres seuls
        if fus and DEBUT_MONTANT.match(fus[-1]) and SUITE_MONTANT.match(x):
            fus[-1] = fus[-1] + ' ' + x
        else:
            fus.append(x)
    vues, propres = set(), []
    for x in fus:
        if BRUIT_LIGNE.match(x) or x in vues:
            continue
        vues.add(x)
        propres.append(x)
    return '\n'.join(propres)

def norm(s):
    s = unicodedata.normalize('NFKC', s).replace('’', "'")
    return re.sub(r'\s+', ' ', s).lower()

def valeurs(e):
    return [(k, e[k]) for k in ('poste', 'entreprise', 'ville', 'remuneration') if e[k]] + [('competence', c) for c in e['competences']]

PROPRES = []
perdues = []
for l in DATA:
    p = dict(l, offre=nettoyer_offre(l['offre']))
    brut, net = norm(l['offre']), norm(p['offre'])
    for k, v in valeurs(l['extraction']):
        if norm(v) in brut and norm(v) not in net:
            perdues.append((l['url'], k, v))
    PROPRES.append(p)
print('Valeurs étiquetées présentes dans le brut mais perdues au nettoyage :', perdues or 0)

for u in [T+'2030285/', 'https://www.keejob.com/offres-emploi/247431/', M+'1283746']:
    p = next(x for x in PROPRES if x['url'] == u)
    net = norm(p['offre'])
    print('-' * 70, '\n', u)
    for k, v in valeurs(p['extraction']):
        print(f"   {'✅' if norm(v) in net else '⚠️ (forme normalisée)'} {k}: {v}")

PROPRE = f'{DOSSIER}/offres_propres.jsonl'
with open(PROPRE, 'w', encoding='utf-8') as f:
    for p in PROPRES:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

import statistics
def stats_tokens(lignes):
    n = [len(tokenizer(x['offre'])['input_ids']) for x in lignes]
    complet = [len(tokenizer(tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': x['offre']},
         {'role': 'assistant', 'content': json.dumps(x['extraction'], ensure_ascii=False)}], tokenize=False))['input_ids']) for x in lignes]
    return statistics.median(n), max(n), max(complet)
avant = [len(tokenizer(x['offre'])['input_ids']) for x in DATA]
med, mx, mx_complet = stats_tokens(PROPRES)
print('-' * 70)
print(f'Tokens par offre (brut)   : médiane {statistics.median(avant)} | max {max(avant)}')
print(f'Tokens par offre (propre) : médiane {med} | max {mx}')
print(f'Exemple complet le plus long (system + offre + JSON) : {mx_complet} tokens')
print('Écrit :', PROPRE, '|', len(PROPRES), 'offres')

### 11. Rééquilibrage (étape 4)

Cette cellule ajoutait au dataset 16 offres copiées à la main (PFE, stages, offres d'emploi en arabe) avec leurs étiquettes.
**Elle n'est pas publiée** : elle contenait le texte complet des offres, qui appartient aux sites d'origine.

In [ ]:
# 12. Écriture finale + bilan (étape 5)
from collections import Counter
FINAL = PROPRES + NOUVELLES
with open(PROPRE, 'w', encoding='utf-8') as f:
    for p in FINAL:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
with open(f'{DOSSIER}/offres_ajouts_brut.jsonl', 'w', encoding='utf-8') as f:   # texte copié avant nettoyage, pour traçabilité
    for a in AJOUTS:
        f.write(json.dumps(a, ensure_ascii=False) + '\n')

with open(PROPRE, encoding='utf-8') as f:
    D = [json.loads(l) for l in f if l.strip()]
for d in D:
    Offre.model_validate(d['extraction'])
    assert d.get('url'), 'url manquante'
med, mx, mx_complet = stats_tokens(D)
print('Fichier         :', PROPRE)
print('Total           :', len(D))
print('Langues         :', dict(Counter(d['extraction']['langue_offre'] for d in D)))
print('Type de contrat :', dict(Counter(d['extraction']['type_contrat'] for d in D)))
print('Sources         :', dict(Counter(d['source'] for d in D)))
print('Doublons        : texte', len(D) - len({norm(d['offre']) for d in D}), '| url', len(D) - len({d['url'] for d in D}))
print('Arabe avec entreprise ET ville :', sum(1 for d in D if d['extraction']['langue_offre'] == 'ar' and d['extraction']['entreprise'] and d['extraction']['ville']))
print(f'Tokens par offre : médiane {med} | max {mx}')
print(f'Exemple complet le plus long (system + offre + JSON) : {mx_complet} tokens')
print('offres.jsonl inchangé :', compter_annotations(), 'lignes')